# Day 25 — Exercises: Cross-Validation, Scaling & Pipelines

**🎯 Objectives:**
1. Compare `KFold` and `StratifiedKFold` class distributions on imbalanced datasets.
2. Build a preprocessing pipeline with `StandardScaler` to prevent data leakage during cross-validation.
3. Use `ColumnTransformer` to handle heterogeneous datasets containing both numerical and categorical features.

## Exercise 1: KFold vs. StratifiedKFold

When evaluating classification models, maintaining the class distribution across cross-validation folds is essential.
- **KFold**: Divides the dataset into $K$ equal-sized folds. If a class is rare (e.g., in highly imbalanced datasets), standard `KFold` may produce folds with very few or zero samples of the minority class, leading to high variance or training errors.
- **StratifiedKFold**: Restructures the splits so that each fold preserves the original percentage of samples for each class. This is the standard practice for classification.

Let's generate an imbalanced dataset and compare the split distributions.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold

# Generate an imbalanced binary classification dataset (90% class 0, 10% class 1)
np.random.seed(42)
n_samples = 150
y = np.array([0] * 135 + [1] * 15)
# Shuffle labels to simulate unsorted data
np.random.shuffle(y)
X = np.random.randn(n_samples, 2)

print(f"Total samples: {len(y)}")
print(f"Overall class counts: {np.bincount(y)}")

def print_splits(splitter, name):
    print(f"\n{'='*10} {name} {'='*10}")
    for fold, (train_idx, val_idx) in enumerate(splitter.split(X, y)):
        y_train, y_val = y[train_idx], y[val_idx]
        train_counts = np.bincount(y_train)
        val_counts = np.bincount(y_val)
        
        # Handle cases where minority class might be missing from a fold count
        train_minority = train_counts[1] if len(train_counts) > 1 else 0
        val_minority = val_counts[1] if len(val_counts) > 1 else 0
        
        print(f"Fold {fold + 1}:")
        print(f"  Train Dist: {train_counts} (Class 1 ratio: {train_minority / len(y_train):.2%})")
        print(f"  Val Dist:   {val_counts} (Class 1 ratio: {val_minority / len(y_val):.2%})")

# 1. KFold without shuffling (problematic if data is ordered)
kf_no_shuffle = KFold(n_splits=5, shuffle=False)
print_splits(kf_no_shuffle, "KFold (shuffle=False)")

# 2. KFold with shuffling
kf_shuffle = KFold(n_splits=5, shuffle=True, random_state=42)
print_splits(kf_shuffle, "KFold (shuffle=True)")

# 3. StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print_splits(skf, "StratifiedKFold")

Total samples: 150
Overall class counts: [135  15]

========== KFold (shuffle=False) ==========
Fold 1:
  Train Dist: [108  12] (Class 1 ratio: 10.00%)
  Val Dist:   [27  3] (Class 1 ratio: 10.00%)
Fold 2:
  Train Dist: [108  12] (Class 1 ratio: 10.00%)
  Val Dist:   [27  3] (Class 1 ratio: 10.00%)
Fold 3:
  Train Dist: [107  13] (Class 1 ratio: 10.83%)
  Val Dist:   [28  2] (Class 1 ratio: 6.67%)
Fold 4:
  Train Dist: [109  11] (Class 1 ratio: 9.17%)
  Val Dist:   [26  4] (Class 1 ratio: 13.33%)
Fold 5:
  Train Dist: [108  12] (Class 1 ratio: 10.00%)
  Val Dist:   [27  3] (Class 1 ratio: 10.00%)

========== KFold (shuffle=True) ==========
Fold 1:
  Train Dist: [108  12] (Class 1 ratio: 10.00%)
  Val Dist:   [27  3] (Class 1 ratio: 10.00%)
Fold 2:
  Train Dist: [108  12] (Class 1 ratio: 10.00%)
  Val Dist:   [27  3] (Class 1 ratio: 10.00%)
Fold 3:
  Train Dist: [111   9] (Class 1 ratio: 7.50%)
  Val Dist:   [24  6] (Class 1 ratio: 20.00%)
Fold 4:
  Train Dist: [107  13] (Class 1 ratio:

## Exercise 2: StandardScaler inside a Pipeline vs. Outside (Data Leakage)

Data leakage is one of the most common validation errors. If we fit a scaler (like `StandardScaler`) on the entire dataset and *then* split or cross-validate, the scaler leaks validation information (the mean and variance of the validation folds) into the training folds.

To run a clean cross-validation:
1. Build a `Pipeline` containing the scaler and model.
2. Pass the `Pipeline` directly to `cross_val_score` or `cross_validate`.

Let's compare the incorrect way (data leakage) with the correct way (pipeline).

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# Load Breast Cancer dataset
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target

# --- Method 1: INCORRECT (Data Leakage) ---
# Fit and transform the scaler on the FULL dataset before splitting/cross-validating
scaler_leak = StandardScaler()
X_bc_scaled_leaked = scaler_leak.fit_transform(X_bc)
model_leak = LogisticRegression(max_iter=10000, random_state=42)
scores_leaked = cross_val_score(model_leak, X_bc_scaled_leaked, y_bc, cv=5)

# --- Method 2: CORRECT (No Leakage using Pipeline) ---
# Define a pipeline that contains both the scaler and model. 
# During cross-validation, the pipeline fits the scaler strictly on the training fold.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=10000, random_state=42))
])
scores_clean = cross_val_score(pipeline, X_bc, y_bc, cv=5)

print("--- Cross-Validation Scores Comparison ---")
print(f"Incorrect CV Scores (with Leakage): {scores_leaked}")
print(f"Incorrect Mean CV Score:           {scores_leaked.mean():.6f}")
print(f"Correct CV Scores (no Leakage):    {scores_clean}")
print(f"Correct Mean CV Score:             {scores_clean.mean():.6f}")

--- Cross-Validation Scores Comparison ---
Incorrect CV Scores (with Leakage): [0.98245614 0.98245614 0.97368421 0.97368421 0.99115044]
Incorrect Mean CV Score:           0.980686
Correct CV Scores (no Leakage):    [0.98245614 0.98245614 0.97368421 0.97368421 0.99115044]
Correct Mean CV Score:             0.980686


## Exercise 3: ColumnTransformer for Heterogeneous Data

Real-world data is rarely purely numerical. We often have text/categorical attributes mixed with continuous metrics.
- **Numerical columns** need imputation (e.g. median) and standard scaling.
- **Categorical columns** need imputation (e.g. mode) and encoding (e.g. `OneHotEncoder`).

We use `ColumnTransformer` to define preprocessing pathways for each type of feature, keeping our code clean and modular.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate

# Generate a toy dataset with numeric and categorical features and missing values
data_dict = {
    'age': [24, 45, np.nan, 36, 54, 22, 40, 61, np.nan, 29],
    'salary': [48000, 85000, 60000, np.nan, 115000, 42000, np.nan, 140000, 75000, 56000],
    'education': ['Bachelor', 'PhD', 'Master', 'Bachelor', 'PhD', 'Bachelor', np.nan, 'Master', 'PhD', 'Bachelor'],
    'city': ['NY', 'SF', 'NY', 'SF', 'SF', 'NY', 'SF', 'NY', np.nan, 'NY'],
    'churn': [0, 1, 1, 0, 1, 0, 1, 1, 0, 0]
}

df = pd.DataFrame(data_dict)
X_toy = df.drop(columns=['churn'])
y_toy = df['churn']

print("--- Original Toy Dataset ---")
print(df)

# Separate feature types
numeric_cols = ['age', 'salary']
categorical_cols = ['education', 'city']

# 1. Preprocessing for Numerical Data
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Preprocessing for Categorical Data
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Combine using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# 4. Build final Pipeline with estimator
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Fit and inspect output shape transformation
full_pipeline.fit(X_toy, y_toy)
processed_X = preprocessor.transform(X_toy)
print(f"\nOriginal shape: {X_toy.shape} -> Processed shape: {processed_X.shape}")

# Perform CV on the full pipeline
cv_results = cross_validate(
    full_pipeline, X_toy, y_toy, 
    cv=3, 
    scoring=['accuracy', 'f1'],
    return_train_score=True
)

print("\n--- Cross-Validation on Toy Dataset ---")
print(f"Validation Accuracies: {cv_results['test_accuracy']}")
print(f"Mean Validation Accuracy: {cv_results['test_accuracy'].mean():.4f}")
print(f"Validation F1-scores: {cv_results['test_f1']}")
print(f"Mean Validation F1-score: {cv_results['test_f1'].mean():.4f}")

--- Original Toy Dataset ---
    age    salary education city  churn
0  24.0   48000.0  Bachelor   NY      0
1  45.0   85000.0       PhD   SF      1
2   NaN   60000.0    Master   NY      1
3  36.0       NaN  Bachelor   SF      0
4  54.0  115000.0       PhD   SF      1
5  22.0   42000.0  Bachelor   NY      0
6  40.0       NaN       NaN   SF      1
7  61.0  140000.0    Master   NY      1
8   NaN   75000.0       PhD  NaN      0
9  29.0   56000.0  Bachelor   NY      0

Original shape: (10, 4) -> Processed shape: (10, 7)



--- Cross-Validation on Toy Dataset ---
Validation Accuracies: [0.5        0.66666667 0.66666667]
Mean Validation Accuracy: 0.6111
Validation F1-scores: [0.5        0.66666667 0.66666667]
Mean Validation F1-score: 0.6111
